
## 1. Setup

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path
from sklearn.linear_model import Ridge
from sklearn.isotonic import IsotonicRegression

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

DATA_DIR = Path("..")
TRAIN_PATH = DATA_DIR / "train/ecommerce_price_prediction-train.csv"
TEST_PATH  = DATA_DIR / "test/ecommerce_price_prediction-test-3-days.csv"
SUBMISSION_PATH = Path("./submission.csv")

ID_COLS   = ["shopId", "itemId", "modelId", "cat_id", "brand"]
BOOL_COLS = ["is_free_shipping", "is_pre_order", "is_official_shop",
             "is_verified", "is_preferred_plus_seller"]
KEY = ["shopId", "itemId", "modelId"]

## 2. Load & clean

In [ ]:
def basic_clean(df):
    df = df.copy()
    df["capturedAt"] = pd.to_datetime(df["capturedAt"], errors="coerce", utc=True)
    for c in BOOL_COLS:
        if c in df.columns:
            df[c] = df[c].map({"t": 1, "f": 0, True: 1, False: 0})
            df[c] = pd.to_numeric(df[c], errors="coerce")
    if "priceBeforeDiscount" in df.columns:
        df.loc[df["priceBeforeDiscount"] == 0, "priceBeforeDiscount"] = np.nan
    return df

train_full = basic_clean(pd.read_csv(TRAIN_PATH))
test       = basic_clean(pd.read_csv(TEST_PATH))

# Hold out the last 3 days as validation (simulates an outage)
N_VAL_DAYS = 3
all_days   = sorted(train_full["capturedAt"].dt.floor("D").unique())
val_days   = all_days[-N_VAL_DAYS:]
val_mask   = train_full["capturedAt"].dt.floor("D").isin(val_days)
validation = train_full[val_mask].reset_index(drop=True)
train      = train_full[~val_mask].reset_index(drop=True)

print(f"Train: {train.shape}, Validation: {validation.shape}, Test: {test.shape}")
print(f"Validation days: {[str(d.date()) for d in val_days]}")


In [ ]:
def detect_price_outliers_per_product(df, key=KEY,
                                       z_thresh=5.0,
                                       min_obs=5,
                                       min_cluster=3,
                                       min_pct_deviation=0.30,
                                       max_recurrence=2,
                                       end_run_min=2):
    """
    Flag genuinely anomalous price observations.

    A row is flagged only if all of the following hold:
      1. STATISTICALLY EXTREME — MAD z-score > z_thresh.
      2. TEMPORALLY ISOLATED — not part of a run of >= min_cluster.
      3. MAGNITUDE-EXTREME — deviation > min_pct_deviation of median.
      4. NON-RECURRING — the deviating value doesn't recur > max_recurrence times.
      5. NOT AN END-OF-SERIES RUN — if the trailing observations form a
         consecutive run of >= end_run_min flagged points, they are
         protected (likely a change-point starting). A single isolated
         spike at the very end is still flagged.

    Parameters
    ----------
    end_run_min : int
        Minimum length of a consecutive flagged run at the END of the
        series for the points to be protected from flagging. The points
        are only protected if the run touches the last observation.
        end_run_min=2 means: 2+ consecutive flagged points at the tail
        are treated as a possible change-point. A single stray spike at
        the tail remains flagged.
    """
    flags = pd.Series(False, index=df.index)
    summary_rows = []

    work = df[key + ["capturedAt", "price"]].copy()
    work["_idx"] = df.index
    work = work.sort_values(key + ["capturedAt"])

    for keys, g in work.groupby(key, sort=False):
        if len(g) < min_obs:
            continue

        prices = g["price"].values
        median = np.median(prices)
        mad    = np.median(np.abs(prices - median))

        # (1) statistical
        if mad == 0:
            stat_mask = np.abs(prices - median) > 0.01 * abs(median)
        else:
            z = 0.6745 * (prices - median) / mad
            stat_mask = np.abs(z) > z_thresh

        # (2) temporal isolation
        isolated_mask = np.zeros_like(stat_mask)
        i = 0
        while i < len(stat_mask):
            if stat_mask[i]:
                j = i
                while j < len(stat_mask) and stat_mask[j]:
                    j += 1
                if (j - i) < min_cluster:
                    isolated_mask[i:j] = True
                i = j
            else:
                i += 1

        # (3) magnitude
        if abs(median) > 0:
            magnitude_mask = np.abs(prices - median) / abs(median) > min_pct_deviation
        else:
            magnitude_mask = np.zeros_like(stat_mask)

        # (4) non-recurrence
        recurrence_mask = np.ones_like(stat_mask)
        for k in np.where(isolated_mask & magnitude_mask)[0]:
            target = prices[k]
            tol = max(abs(target) * 0.01, 1.0)
            n_similar = np.sum(np.abs(prices - target) <= tol)
            if n_similar > max_recurrence:
                recurrence_mask[k] = False

        # Provisional flags before end-of-series protection
        provisional_mask = isolated_mask & magnitude_mask & recurrence_mask

        # (5) END-OF-SERIES PROTECTION
        end_safe_mask = np.ones_like(provisional_mask)
        n = len(provisional_mask)
        if n > 0 and provisional_mask[-1]:
            # Find the start of the trailing run
            run_start = n - 1
            while run_start > 0 and provisional_mask[run_start - 1]:
                run_start -= 1
            run_len = n - run_start
            if run_len >= end_run_min:
                end_safe_mask[run_start:] = False  # protect the whole run

        final_mask = provisional_mask & end_safe_mask

        n_out = int(final_mask.sum())
        if n_out > 0:
            flags.loc[g["_idx"].values[final_mask]] = True
            row = {f"{k}": v for k, v in zip(key, keys if isinstance(keys, tuple) else (keys,))}
            row.update({
                "n_obs":             len(g),
                "median_price":      float(median),
                "mad":               float(mad),
                "n_stat":            int(stat_mask.sum()),
                "n_after_isolation": int(isolated_mask.sum()),
                "n_after_magnitude": int((isolated_mask & magnitude_mask).sum()),
                "n_provisional":     int(provisional_mask.sum()),
                "n_outliers":        n_out,
            })
            summary_rows.append(row)

    summary = (pd.DataFrame(summary_rows).sort_values("n_outliers", ascending=False)
                                          .reset_index(drop=True)
               if summary_rows else
               pd.DataFrame(columns=key + ["n_obs","median_price","mad",
                                            "n_stat","n_after_isolation",
                                            "n_after_magnitude","n_provisional",
                                            "n_outliers"]))
    return summary, flags

print("PER-PRODUCT PRICE OUTLIERS (MAD z-score, |z| > 5)")
prod_summary, prod_flags = detect_price_outliers_per_product(train, z_thresh=5.0)
print(f"Products with at least one outlier: {len(prod_summary):,}")
print(f"Total outlier rows:                 {prod_flags.sum():,} "
      f"({prod_flags.mean()*100:.3f}%)")

if len(prod_summary):
    print("\nTop 10 products by outlier count:")
    print(prod_summary.head(10).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def plot_top_outlier_products(df, prod_summary, prod_flags,
                              key=KEY, top_n=10, ncols=2,
                              figsize_per_plot=(7, 3)):
    """
    Plot price-over-time for the top-N products with the most flagged
    outlier observations.

    Parameters
    ----------
    df            : original DataFrame containing capturedAt, price, and key cols
    prod_summary  : DataFrame returned by detect_price_outliers_per_product
    prod_flags    : Boolean Series returned by detect_price_outliers_per_product
    key           : product identifier columns
    top_n         : how many products to plot
    ncols         : grid columns
    figsize_per_plot : (w, h) per subplot
    """
    if len(prod_summary) == 0:
        print("No flagged products to plot.")
        return

    top = prod_summary.head(top_n)
    n   = len(top)
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(figsize_per_plot[0] * ncols, figsize_per_plot[1] * nrows),
        squeeze=False,
    )
    axes = axes.flatten()

    for i, (_, prod) in enumerate(top.iterrows()):
        ax = axes[i]

        # Pull this product's full history
        sub = df.copy()
        for k in key:
            sub = sub[sub[k] == prod[k]]
        sub = sub.sort_values("capturedAt")

        if sub.empty:
            ax.set_visible(False)
            continue

        # Identify flagged rows for this product
        flagged_idx = prod_flags.index[prod_flags]
        is_outlier  = sub.index.isin(flagged_idx)

        # Plot all observations as a line + scatter
        ax.plot(sub["capturedAt"], sub["price"],
                color="steelblue", alpha=0.5, linewidth=1, zorder=1)
        ax.scatter(sub.loc[~is_outlier, "capturedAt"],
                   sub.loc[~is_outlier, "price"],
                   color="steelblue", s=20, label="normal", zorder=2)
        ax.scatter(sub.loc[is_outlier, "capturedAt"],
                   sub.loc[is_outlier, "price"],
                   color="crimson", s=45, marker="X",
                   label="flagged", zorder=3, edgecolor="black", linewidth=0.6)

        # Reference line at the product median
        ax.axhline(prod["median_price"], color="gray", linestyle="--",
                   linewidth=0.8, alpha=0.7, zorder=0)

        # Compact title
        item_label = f"item {prod['itemId']} / model {prod['modelId']}"
        ax.set_title(
            f"shop {prod['shopId']} — {item_label}\n"
            f"n_obs={prod['n_obs']}, "
            f"median={prod['median_price']:,.0f}, "
            f"outliers={prod['n_outliers']}",
            fontsize=9,
        )
        ax.set_xlabel("")
        ax.set_ylabel("price")
        ax.tick_params(axis="x", rotation=30, labelsize=8)
        ax.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=5))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
        ax.grid(True, alpha=0.3)
        if i == 0:
            ax.legend(loc="best", fontsize=8, framealpha=0.9)

    # Hide unused subplot slots
    for j in range(n, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(f"Top {n} products by flagged-outlier count",
                 fontsize=12, y=1.00)
    plt.tight_layout()
    plt.show()


# %% Run it
plot_top_outlier_products(train, prod_summary, prod_flags, top_n=10, ncols=2)

In [ ]:
# %%
def remove_flagged_outliers(df, flags, verbose=True):
    """
    Drop rows flagged as outliers and return a clean copy.

    Parameters
    ----------
    df       : DataFrame to clean
    flags    : Boolean Series aligned with df.index. True = drop.
    verbose  : if True, print before/after row counts

    Returns
    -------
    cleaned_df : DataFrame with flagged rows removed (index reset)
    """
    # Align flags to df just in case the indices have drifted
    flags = flags.reindex(df.index, fill_value=False)

    n_before = len(df)
    n_dropped = int(flags.sum())
    cleaned_df = df.loc[~flags].reset_index(drop=True)
    n_after = len(cleaned_df)

    if verbose:
        pct = (n_dropped / n_before * 100) if n_before else 0.0
        print(f"Rows before:  {n_before:,}")
        print(f"Rows dropped: {n_dropped:,} ({pct:.4f}%)")
        print(f"Rows after:   {n_after:,}")

    return cleaned_df

train = remove_flagged_outliers(train, prod_flags)

## 3. Shared utilities — temporal features & calibrator

In [ ]:
def add_temporal_features(df):
    df = df.copy()
    t = df["capturedAt"]
    df["dow"]            = t.dt.dayofweek.astype("int8")
    df["dom"]            = t.dt.day.astype("int8")
    df["month"]          = t.dt.month.astype("int8")
    df["is_weekend"]     = (t.dt.dayofweek >= 5).astype("int8")
    df["is_double_date"] = (t.dt.day == t.dt.month).astype("int8")
    return df

def fit_calibrator(anchors_pred, anchors_true, cat_ids=None, k_shrink=5.0):
    """
    Two-layer anchor calibrator (returns a callable):
      L1: global multiplicative bias (median log-residual on anchors)
      L2: per-category bias with empirical-Bayes shrinkage toward L1
    """
    ap = np.asarray(anchors_pred, dtype=float)
    at = np.asarray(anchors_true, dtype=float)
    valid = np.isfinite(ap) & np.isfinite(at) & (ap > 0) & (at > 0)
    ap, at = ap[valid], at[valid]
    cids   = np.asarray(cat_ids)[valid] if cat_ids is not None else None

    if len(ap) < 5:
        return (lambda raw, cat_id_array=None: np.asarray(raw, dtype=float),
                {"global_bias": 0.0, "n_cat": 0})

    log_resid   = np.log(at / ap)
    global_bias = np.median(log_resid)

    cat_map = {}
    if cids is not None:
        df = pd.DataFrame({"cat": cids, "lr": log_resid})
        for cat, g in df.groupby("cat"):
            n  = len(g)
            mu = np.median(g["lr"])
            if np.isfinite(mu):
                cat_map[cat] = (n * mu + k_shrink * global_bias) / (n + k_shrink)

    def apply(raw, cat_id_array=None):
        raw = np.asarray(raw, dtype=float)
        if cat_id_array is not None and cat_map:
            corr = np.array([np.exp(cat_map.get(c, global_bias)) for c in cat_id_array])
            return raw * corr
        return raw * np.exp(global_bias)

    return apply, {"global_bias": global_bias, "n_cat": len(cat_map)}


def metrics(pred, true):
    pred = np.asarray(pred, dtype=float)
    true = np.asarray(true, dtype=float)
    valid = np.isfinite(pred) & np.isfinite(true) & (pred > 0) & (true > 0)
    p, t  = pred[valid], true[valid]
    if len(p) == 0:
        return dict(n=0, MAE=np.nan, RMSE=np.nan, MAPE=np.nan, MedAPE=np.nan)
    return dict(
        n=valid.sum(),
        MAE=np.mean(np.abs(p - t)),
        RMSE=np.sqrt(np.mean((p - t) ** 2)),
        MAPE=np.mean(np.abs(p - t) / t) * 100,
        MedAPE=np.median(np.abs(p - t) / t) * 100,
    )

def inject_anchor_state(target_df, anchors_df, history_df, k_shrink=2, include_cat_disc=True):
    target_df = target_df.copy()
    if "cat_id_hist" not in target_df.columns:
        cat_lookup = (history_df.sort_values("capturedAt")
                      .groupby(KEY)["cat_id"].last().reset_index()
                      .rename(columns={"cat_id":"cat_id_hist"}))
        target_df = target_df.merge(cat_lookup, on=KEY, how="left")

    day_avg_disc = anchors_df["show_discount"].mean()
    target_df["day_avg_discount"] = day_avg_disc
    target_df["day_promo_rate"]   = (anchors_df["promotionId"].fillna(0) != 0).mean()
    target_df["day_free_ship"]    = anchors_df["is_free_shipping"].mean()
    
    if include_cat_disc:
        cat = (anchors_df.groupby("cat_id")
       .agg(cat_disc=("show_discount","mean"), cat_n=("show_discount","size"))
       .reset_index().rename(columns={"cat_id":"cat_id_hist"}))

        # Use the median across categories as the shrinkage target,
        # not the day-wide mean of all anchors
        cat_disc_target = cat["cat_disc"].median()              # robust target
        
        cat["cat_disc_shrunk"] = (
            cat["cat_n"] * cat["cat_disc"].fillna(cat_disc_target)
            + k_shrink * cat_disc_target
        ) / (cat["cat_n"] + k_shrink)

        target_df = target_df.merge(cat[["cat_id_hist","cat_disc_shrunk"]],
                                    on="cat_id_hist", how="left")
        target_df["cat_disc_shrunk"] = target_df["cat_disc_shrunk"].fillna(day_avg_disc)
    return target_df

 ##  4. Approach 1

In [ ]:

# %%
def fit_global_bias(anchor_true, anchor_pred):
    """
    Estimate a single multiplicative bias correction from anchors.
    Returns the median log-residual: log(true / pred).
    A positive value means the model is under-predicting on average.
    """
    at = np.asarray(anchor_true, dtype=float)
    ap = np.asarray(anchor_pred, dtype=float)
    valid = np.isfinite(at) & np.isfinite(ap) & (at > 0) & (ap > 0)
    if valid.sum() < 5:
        return 0.0
    return float(np.median(np.log(at[valid] / ap[valid])))


def fit_per_cat_bias(anchor_true, anchor_pred, anchor_cat,
                     global_bias, k_shrink=5.0):
    """
    Estimate per-category log-residual biases with empirical-Bayes shrinkage
    toward the global bias.

    For each category:
        cat_bias = (n * cat_median + k * global_bias) / (n + k)

    Categories with few anchors get pulled hard toward the global bias;
    categories with many anchors keep their own estimate.

    Returns a dict {cat_id: shrunken_log_bias}.
    """
    at  = np.asarray(anchor_true, dtype=float)
    ap  = np.asarray(anchor_pred, dtype=float)
    cat = np.asarray(anchor_cat)

    valid = np.isfinite(at) & np.isfinite(ap) & (at > 0) & (ap > 0)
    if valid.sum() < 5:
        return {}

    log_resid = np.log(at[valid] / ap[valid])
    df = pd.DataFrame({"cat": cat[valid], "lr": log_resid})

    cat_map = {}
    for c, g in df.groupby("cat"):
        n  = len(g)
        mu = np.median(g["lr"])
        if np.isfinite(mu):
            cat_map[c] = (n * mu + k_shrink * global_bias) / (n + k_shrink)
    return cat_map


def calibrate(anchor_true, anchor_pred, prediction_output,
              method="global",
              anchor_cat=None, target_cat=None,
              k_shrink=5.0):
    """
    Calibrate raw model predictions using anchor signals.

    Parameters
    ----------
    anchor_true       : array-like, true prices on anchor rows
    anchor_pred       : array-like, raw model predictions on anchor rows
    prediction_output : array-like, raw model predictions on target rows (to be calibrated)
    method            : "global" or "global_plus_cat"
    anchor_cat        : array-like of category ids for each anchor row
                        (required when method == "global_plus_cat")
    target_cat        : array-like of category ids for each target row
                        (required when method == "global_plus_cat")
    k_shrink          : EB shrinkage strength for per-category bias

    Returns
    -------
    calibrated_prediction : np.ndarray, same length as prediction_output
    diagnostics           : dict with global_bias, n_cat, method
    """
    raw = np.asarray(prediction_output, dtype=float)

    # Layer 1: global bias (always applied)
    global_bias = fit_global_bias(anchor_true, anchor_pred)

    if method == "global":
        calibrated = raw * np.exp(global_bias)
        return calibrated, {"method": method,
                            "global_bias": global_bias,
                            "n_cat": 0}

    elif method == "global_plus_cat":
        if anchor_cat is None or target_cat is None:
            raise ValueError("method='global_plus_cat' requires anchor_cat and target_cat")

        cat_map = fit_per_cat_bias(anchor_true, anchor_pred, anchor_cat,
                                   global_bias=global_bias, k_shrink=k_shrink)

        # For each target row, look up its category bias; fall back to global
        target_cat = np.asarray(target_cat)
        bias_arr = np.array([cat_map.get(c, global_bias) for c in target_cat])
        calibrated = raw * np.exp(bias_arr)

        return calibrated, {"method": method,
                            "global_bias": global_bias,
                            "n_cat": len(cat_map)}

    else:
        raise ValueError(f"Unknown method: {method!r}. "
                         "Use 'global' or 'global_plus_cat'.")

In [ ]:
A1_NUM_FEATS = [
    # Temporal features
    "dow", "dom", "month", "is_weekend", "is_double_date",
    # Anchor-derived day state
    "day_avg_discount", "day_promo_rate", "day_free_ship",
]
A1_CAT_FEATS = ["shopId", "itemId", "modelId"]
A1_ALL_FEATS = A1_NUM_FEATS + A1_CAT_FEATS


def build_a1_training_set(train_df, n_anchors=100, min_day_size=101, seed=SEED):
    """
    Build the Approach 1 training set with simulated same-day anchors.

    For each historical day:
      1. Sample 100 rows as 'simulated anchors'.
      2. Use the rest as training targets.
      3. Compute anchor-derived day-state features from the simulated
         anchors (day_avg_discount, cat_disc_shrunk, etc.).
      4. Inject those features as columns on the targets.
      5. Add temporal features.

    This makes training symmetric with inference: at test time the real
    100 anchors will provide the same kind of features.
    """
    rng = np.random.default_rng(seed)
    df = train_df.copy()
    df["day"] = df["capturedAt"].dt.floor("D")
    days = sorted(df["day"].unique())

    parts = []
    for i, day in enumerate(days):
        day_rows = df[df["day"] == day]
        if len(day_rows) < min_day_size:
            continue

        # Sample simulated anchors for this day
        anchor_idx = rng.choice(day_rows.index,
                                size=min(n_anchors, len(day_rows) // 2),
                                replace=False)
        anchors = day_rows.loc[anchor_idx]
        targets = day_rows.drop(anchor_idx)

        targets = targets.rename(columns={"cat_id": "cat_id_hist"})

        targets = inject_anchor_state(targets, anchors, df, include_cat_disc=False)
        targets = add_temporal_features(targets)
        parts.append(targets)

        if (i + 1) % 5 == 0 or (i + 1) == len(days):
            print(f"  day {i+1}/{len(days)}, accumulated rows: {sum(len(p) for p in parts):,}")

    return pd.concat(parts, ignore_index=True)


def prepare_a1(df):
    """Cast categorical features and select the final feature set."""
    df = df.copy()
    for c in A1_CAT_FEATS:
        if c in df.columns:
            df[c] = df[c].astype("category")
    return df

def train_approach1(train_df):
    feat = build_a1_training_set(train_df)
    feat = feat.dropna(subset=["price"])
    feat = feat[feat["price"] > 0]
    feat["y"] = np.log(feat["price"])  # predict log-price directly

    feat = feat.sort_values("capturedAt")
    cut = feat["capturedAt"].quantile(0.9)
    tr  = feat[feat["capturedAt"] <= cut]
    vl  = feat[feat["capturedAt"] >  cut]

    tr = prepare_a1(tr); vl = prepare_a1(vl)

    # Confirm all required features exist
    missing = [c for c in A1_ALL_FEATS if c not in tr.columns]
    assert not missing, f"Missing features in training set: {missing}"

    dtr = lgb.Dataset(tr[A1_ALL_FEATS], tr["y"], categorical_feature=A1_CAT_FEATS)
    dvl = lgb.Dataset(vl[A1_ALL_FEATS], vl["y"], categorical_feature=A1_CAT_FEATS,
                      reference=dtr)

    params = dict(
        objective="regression_l1", metric="mae",
        learning_rate=0.05, num_leaves=127, min_data_in_leaf=50,
        feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
        lambda_l2=1.0, seed=SEED, verbose=-1,
    )
    model = lgb.train(
        params, dtr, num_boost_round=2000,
        valid_sets=[dtr, dvl], valid_names=["train", "val"],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)],
    )
    return model


def predict_approach1(model, target_df, anchors_df, history_df):
    """
    Predict for target_df using same-day anchors to compute day-state features.

    Steps:
      1. Inject anchor state (day_avg_discount, cat_disc_shrunk, etc.).
      2. Add temporal features.
      3. Cast categoricals.
      4. Predict log-price → exponentiate.
    """
    df = target_df.copy()
    if "cat_id_hist" not in df.columns and "cat_id" in df.columns:
        df = df.rename(columns={"cat_id": "cat_id_hist"})

    df = inject_anchor_state(df, anchors_df, history_df)
    df = add_temporal_features(df)
    df = prepare_a1(df)

    # Ensure every expected feature exists
    for c in A1_ALL_FEATS:
        if c not in df.columns:
            df[c] = np.nan

    pred_log = model.predict(df[A1_ALL_FEATS], num_iteration=model.best_iteration)
    return np.exp(pred_log)


def evaluate_approach1(model, train_df, validation_df, n_anchors=100, seed=SEED):
    """Evaluate A1 on held-out validation days with realistic anchor splits."""
    rng = np.random.default_rng(seed)
    val = validation_df.copy()
    val["day"] = val["capturedAt"].dt.floor("D")

    parts = []
    for day in sorted(val["day"].unique()):
        day_rows = val[val["day"] == day]
        anchor_idx = rng.choice(day_rows.index, size=n_anchors, replace=False)
        anchors = day_rows.loc[anchor_idx]
        targets = day_rows.drop(anchor_idx)

        # Predict using real anchors as the day-state source
        raw_t = predict_approach1(model, targets, anchors, train_df)
        raw_a = predict_approach1(model, anchors, anchors, train_df)

        # Anchor calibration (L1 + L2, from our refactored calibrate function)
        cal_t, diag = calibrate(
            anchor_true=anchors["price"].values,
            anchor_pred=raw_a,
            prediction_output=raw_t,
            method="global",
            anchor_cat=anchors["cat_id"].values,
            target_cat=targets["cat_id"].values,
        )

        parts.append(pd.DataFrame({
            "day":         day,
            "true_price":  targets["price"].values,
            "pred_raw":    raw_t,
            "pred_cal":    cal_t,
            "global_bias": diag["global_bias"],
        }))
    return pd.concat(parts, ignore_index=True)

# %% Run Approach 1
print("APPROACH 1 — Global Model (IDs + temporal + anchor state)")
t0 = time.time()
model_a1 = train_approach1(train)
eval_a1  = evaluate_approach1(model_a1, train, validation)
print(f"\nA1 raw:        {metrics(eval_a1['pred_raw'], eval_a1['true_price'])}")
print(f"A1 calibrated: {metrics(eval_a1['pred_cal'], eval_a1['true_price'])}")
print(f"Runtime: {time.time()-t0:.1f}s")

## 5. Approach 2 

In [ ]:
RIDGE_NUM_FEATS = [
    # Temporal
    "dow", "dom", "month", "is_weekend", "is_double_date",
    # Anchor-derived day state
    "day_avg_discount", "day_promo_rate", "day_free_ship", "cat_disc_shrunk",
]

def build_a2_training_set(train_df, n_anchors=100, min_day_size=101, seed=SEED):
    """
    Build the per-entity training set with simulated same-day anchor features.

    For each historical day:
      1. Sample 100 rows as 'simulated anchors'.
      2. Compute anchor-derived day state from those anchors.
      3. Stamp the day-state features onto every row of that day.

    Returns the augmented training DataFrame, ready for fit_entity_models.
    """
    rng = np.random.default_rng(seed)
    df = train_df.copy()
    df["day"] = df["capturedAt"].dt.floor("D")
    df = add_temporal_features(df)

    days = sorted(df["day"].unique())
    parts = []
    for i, day in enumerate(days):
        day_rows = df[df["day"] == day]
        if len(day_rows) < min_day_size:
            continue

        # Simulate same-day anchors
        anchor_idx = rng.choice(day_rows.index,
                                size=min(n_anchors, len(day_rows) // 2),
                                replace=False)
        anchors = day_rows.loc[anchor_idx]

        # Inject day state onto every row of this day
        day_with_state = day_rows.rename(columns={"cat_id": "cat_id_hist"})
        day_with_state = inject_anchor_state(day_with_state, anchors, df)
        parts.append(day_with_state)

        if (i + 1) % 5 == 0 or (i + 1) == len(days):
            print(f"  day {i+1}/{len(days)}, accumulated rows: {sum(len(p) for p in parts):,}")

    return pd.concat(parts, ignore_index=True)

def fit_entity_models(train_with_state, group_key, min_obs=5, alpha=1.0):
    """
    Fit one Ridge per entity using the anchor-augmented training set.

    The entity grouping is implicit — we don't pass shop/item/cat IDs as
    features because each Ridge is fit on observations from a single
    entity by construction.

    Parameters
    ----------
    train_with_state : DataFrame from build_a2_training_set
    group_key        : columns defining an entity (e.g. ["cat_id"])
    min_obs          : minimum observations per entity
    alpha            : L2 regularisation strength
    """
    # Use cat_id_hist if grouping by cat_id (build_a2_training_set renamed it)
    actual_group_key = ["cat_id_hist" if k == "cat_id" else k for k in group_key]

    df = train_with_state.dropna(subset=["price"])
    df = df[df["price"] > 0]

    models = {}
    for keys, g in df.groupby(actual_group_key):
        if not isinstance(keys, tuple):
            keys = (keys,)
        if len(g) < min_obs:
            continue
        X = g[RIDGE_NUM_FEATS].fillna(0).values
        y = np.log(g["price"].clip(lower=1e-3).values)
        m = Ridge(alpha=alpha).fit(X, y)
        # Store per-entity price range to clip predictions to a sane range
        models[keys] = (m, g["price"].min(), g["price"].max())
    return models

def predict_entity(models, target_df, anchors_df, history_df, group_key):
    """
    Predict prices for target_df using:
      - per-entity Ridge models (fit on history with simulated anchors)
      - same-day real anchors to compute day-state features

    Falls back to NaN for entities with no fitted model — these can be
    handled at the call site (e.g. by falling back to A1's global model).
    """
    df = target_df.copy()
    if "cat_id_hist" not in df.columns and "cat_id" in df.columns:
        df = df.rename(columns={"cat_id": "cat_id_hist"})

    df = add_temporal_features(df)

    # Inject same-day anchor state onto target rows
    df = inject_anchor_state(df, anchors_df, history_df)

    # Translate "cat_id" in group_key to "cat_id_hist" for lookup
    actual_group_key = ["cat_id_hist" if k == "cat_id" else k for k in group_key]

    preds = np.full(len(df), np.nan)
    feat_arr = df[RIDGE_NUM_FEATS].fillna(0).values
    keys_arr = df[actual_group_key].values

    for i in range(len(df)):
        k = tuple(keys_arr[i])
        if k not in models:
            continue
        m, p_min, p_max = models[k]
        pred = float(np.exp(m.predict(feat_arr[i:i+1])[0]))
        preds[i] = np.clip(pred, p_min * 0.5, p_max * 2.0)
    return preds


def evaluate_approach2(train_df, validation_df, group_key,
                        n_anchors=100, seed=SEED):
    """Evaluate Approach 2 on held-out validation days with realistic anchors."""
    print(f"  building anchor-augmented training set ...")
    train_with_state = build_a2_training_set(train_df, n_anchors=n_anchors, seed=seed)

    print(f"  fitting Ridge models grouped by {group_key} ...")
    models = fit_entity_models(train_with_state, group_key=group_key)
    print(f"  fitted: {len(models):,} entities")

    rng = np.random.default_rng(seed)
    val = validation_df.copy()
    val["day"] = val["capturedAt"].dt.floor("D")

    parts = []
    for day in sorted(val["day"].unique()):
        day_rows = val[val["day"] == day]
        anchor_idx = rng.choice(day_rows.index, size=n_anchors, replace=False)
        anchors = day_rows.loc[anchor_idx]
        targets = day_rows.drop(anchor_idx)

        raw_t = predict_entity(models, targets, anchors, train_df, group_key)
        raw_a = predict_entity(models, anchors, anchors, train_df, group_key)

        cal_t, diag = calibrate(
            anchor_true=anchors["price"].values,
            anchor_pred=raw_a,
            prediction_output=raw_t,
            method="global_plus_cat",
            anchor_cat=anchors["cat_id"].values,
            target_cat=targets["cat_id"].values,
        )

        parts.append(pd.DataFrame({
            "day":         day,
            "true_price":  targets["price"].values,
            "pred_raw":    raw_t,
            "pred_cal":    cal_t,
            "global_bias": diag["global_bias"],
            "n_predicted": np.isfinite(raw_t).sum(),  # how many had a fitted Ridge
        }))
    return models, pd.concat(parts, ignore_index=True)


# %% Run Approach 2
print("\n" + "=" * 60)
print("APPROACH 2 — Per-Entity Ridge Models (same features as A1)")
print("=" * 60)
configs_a2 = [(["shopId", "itemId", "modelId"],  "Per-product")]
results_a2 = {}
for gk, label in configs_a2:
    t0 = time.time()
    print(f"\n--- {label} ---")
    models_a2, res = evaluate_approach2(train, validation, group_key=gk)
    coverage = res["pred_raw"].notna().mean() * 100
    print(f"  coverage:   {coverage:.1f}% of target rows had a fitted model")
    print(f"  raw:        {metrics(res['pred_raw'], res['true_price'])}")
    print(f"  calibrated: {metrics(res['pred_cal'], res['true_price'])}")
    print(f"  runtime: {time.time()-t0:.1f}s")
    results_a2[label] = res

## 6. Approach 3 

- **History-derived features** (last_price, rolling stats, momentum, …)
- **Anchor-injected features** (day-level discount, per-cat shrunken state)
- **Target = log-residual** vs `last_price`
- **2-layer calibration** (global → per-cat)

In [ ]:
def build_history_features(history_df, target_df, key=KEY):
    history_df = history_df.sort_values("capturedAt").reset_index(drop=True)
    target_df  = target_df.copy().reset_index(drop=True)
    target_df["_orig_idx"] = np.arange(len(target_df))

    # (a) asof — last snapshot per product
    asof_left = target_df[["capturedAt"] + key + ["_orig_idx"]].sort_values("capturedAt")
    asof_right = (history_df[["capturedAt"] + key + ["price","show_discount",
                              "promotionId","cat_id","brand"]]
                  .rename(columns={"capturedAt":"last_capturedAt", "price":"last_price",
                                   "show_discount":"last_discount",
                                   "promotionId":"last_promotionId",
                                   "cat_id":"cat_id_hist", "brand":"brand_hist"})
                  .sort_values("last_capturedAt"))
    asof = pd.merge_asof(asof_left, asof_right,
                         left_on="capturedAt", right_on="last_capturedAt",
                         by=key, allow_exact_matches=False, direction="backward")
    asof["days_since_last"] = (asof["capturedAt"] - asof["last_capturedAt"]).dt.total_seconds()/86400
    asof_feats = asof.drop(columns=["capturedAt"] + key)

    # (b) all-time aggregates
    agg = (history_df.groupby(key)["price"]
           .agg(price_mean_all="mean", price_median_all="median", price_std_all="std",
                price_min_all="min", price_max_all="max", n_obs_all="count")
           .reset_index())

    # (c) windowed last-N
    def last_n(df, n, prefix):
        g = (df.groupby(key, group_keys=False)
               .apply(lambda x: x.nlargest(n, "capturedAt")["price"]
                                  .agg(["mean","median","std","min","max"])))
        return g.reset_index().rename(columns={
            "mean":f"{prefix}_mean","median":f"{prefix}_median",
            "std":f"{prefix}_std","min":f"{prefix}_min","max":f"{prefix}_max"})
    last5  = last_n(history_df, 5,  "p_last5")
    last20 = last_n(history_df, 20, "p_last20")

    # (d) discount cadence
    disc = (history_df.assign(_on_sale=(history_df["show_discount"].fillna(0) > 0).astype(int))
            .groupby(key)
            .agg(disc_freq=("_on_sale","mean"), disc_mean=("show_discount","mean"),
                 disc_max=("show_discount","max"))
            .reset_index())

    # (e) shop & cat stats
    shop = (history_df.groupby("shopId")
            .agg(shop_n_listings=("itemId","nunique"), shop_price_median=("price","median"))
            .reset_index())
    cat = (history_df.groupby("cat_id")["price"]
           .agg(cat_median="median", cat_mean="mean").reset_index()
           .rename(columns={"cat_id":"cat_id_hist"}))

    out = (target_df.merge(asof_feats, on="_orig_idx", how="left")
                    .merge(agg,    on=key,           how="left")
                    .merge(last5,  on=key,           how="left")
                    .merge(last20, on=key,           how="left")
                    .merge(disc,   on=key,           how="left")
                    .merge(shop,   on="shopId",      how="left")
                    .merge(cat,    on="cat_id_hist", how="left"))

    out["price_cv_all"]   = out["price_std_all"] / (out["price_mean_all"] + 1e-6)
    out["last_vs_mean"]   = out["last_price"]    / (out["price_mean_all"] + 1e-6)
    out["last_vs_min"]    = out["last_price"]    / (out["price_min_all"]  + 1e-6)
    out["momentum_short"] = out["last_price"]    / (out["p_last5_mean"]   + 1e-6)
    out["momentum_long"]  = out["p_last5_mean"]  / (out["p_last20_mean"]  + 1e-6)

    return out.sort_values("_orig_idx").drop(columns=["_orig_idx"]).reset_index(drop=True)


def build_a3_training_set(train_df, n_anchors=100, min_day_size=101, seed=SEED):
    rng = np.random.default_rng(seed)
    df = train_df.copy()
    df["day"] = df["capturedAt"].dt.floor("D")
    days = sorted(df["day"].unique())
    parts = []
    for i, day in enumerate(days):
        day_rows = df[df["day"] == day]
        if len(day_rows) < min_day_size:
            continue
        hist = df[df["day"] < day]
        if len(hist) < 1000:
            continue
        anchor_idx = rng.choice(day_rows.index,
                                size=min(n_anchors, len(day_rows)//2), replace=False)
        anchors = day_rows.loc[anchor_idx]
        targets = day_rows.drop(anchor_idx)
        feat = build_history_features(hist, targets)
        feat = inject_anchor_state(feat, anchors, hist)
        feat = add_temporal_features(feat)
        parts.append(feat)
        if (i + 1) % 5 == 0 or (i + 1) == len(days):
            print(f"  day {i+1}/{len(days)}, accumulated rows: {sum(len(p) for p in parts):,}")
    return pd.concat(parts, ignore_index=True)

def train_approach3(train_df):
    feat = build_a3_training_set(train_df)
    feat = feat.dropna(subset=["last_price","price"])
    feat = feat[(feat["last_price"] > 0) & (feat["price"] > 0)]
    feat["y"] = np.log(feat["price"] / feat["last_price"])
    feat = feat.sort_values("capturedAt")
    cut = feat["capturedAt"].quantile(0.9)
    tr  = feat[feat["capturedAt"] <= cut]
    vl  = feat[feat["capturedAt"] >  cut]
    for c in A3_CAT_FEATS:
        tr[c] = tr[c].astype("category"); vl[c] = vl[c].astype("category")

    dtr = lgb.Dataset(tr[A3_ALL_FEATS], tr["y"], categorical_feature=A3_CAT_FEATS)
    dvl = lgb.Dataset(vl[A3_ALL_FEATS], vl["y"], categorical_feature=A3_CAT_FEATS, reference=dtr)
    params = dict(objective="regression_l1", metric="mae", learning_rate=0.05,
                  num_leaves=255, min_data_in_leaf=50,
                  feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
                  lambda_l2=1.0, seed=SEED, verbose=-1)
    return lgb.train(params, dtr, num_boost_round=3000,
                     valid_sets=[dtr, dvl], valid_names=["train","val"],
                     callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)])

def evaluate_approach3(model, train_df, validation_df, n_anchors=100, seed=SEED):
    rng = np.random.default_rng(seed)
    val = validation_df.copy()
    val["day"] = val["capturedAt"].dt.floor("D")
    parts = []
    for day in sorted(val["day"].unique()):
        day_rows = val[val["day"] == day]
        anchor_idx = rng.choice(day_rows.index, size=n_anchors, replace=False)
        anchors = day_rows.loc[anchor_idx]
        targets = day_rows.drop(anchor_idx)

        tf = build_history_features(train_df, targets)
        af = build_history_features(train_df, anchors)
        tf = inject_anchor_state(tf, anchors, train_df)
        af = inject_anchor_state(af, anchors, train_df)
        tf = add_temporal_features(tf); af = add_temporal_features(af)
        for c in A3_CAT_FEATS:
            tf[c] = tf[c].astype("category"); af[c] = af[c].astype("category")

        yhat_t = model.predict(tf[A3_ALL_FEATS], num_iteration=model.best_iteration)
        yhat_a = model.predict(af[A3_ALL_FEATS], num_iteration=model.best_iteration)
        raw_t  = tf["last_price"].values * np.exp(yhat_t)
        raw_a  = af["last_price"].values * np.exp(yhat_a)

        # L1 + L2
        valid_a = np.isfinite(raw_a) & np.isfinite(anchors["price"].values) & (raw_a > 0)
        log_resid = np.log(anchors["price"].values[valid_a] / raw_a[valid_a])
        gb = np.median(log_resid)
        pred_l1 = raw_t * np.exp(gb)

        cat_map = {}
        df_cat = pd.DataFrame({"cat": af["cat_id_hist"].values[valid_a], "lr": log_resid})
        for c, g in df_cat.groupby("cat"):
            mu = np.median(g["lr"]); n = len(g)
            if np.isfinite(mu):
                cat_map[c] = (n*mu + 5.0*gb) / (n + 5.0)
        corr = np.array([np.exp(cat_map.get(c, gb)) for c in tf["cat_id_hist"].values])
        pred_l2 = raw_t * corr

        # L3 isotonic
        nan_mask = ~np.isfinite(raw_t) | (raw_t <= 0)
        iso = IsotonicRegression(out_of_bounds="clip", increasing=True)
        iso.fit(raw_a[valid_a] * np.exp(gb), anchors["price"].values[valid_a])
        safe_l2 = np.where(nan_mask, 1.0, pred_l2)
        pred_l3 = np.where(nan_mask, np.nan, iso.predict(safe_l2))

        parts.append(pd.DataFrame({
            "day": day, "true_price": targets["price"].values,
            "pred_raw": raw_t, "pred_l1": pred_l1,
            "pred_l2": pred_l2, "pred_l3": pred_l3,
        }))
    return pd.concat(parts, ignore_index=True)


print("\n" + "=" * 60)
print("APPROACH 3 — Full Pipeline")
print("=" * 60)

A3_NUM_FEATS = [
    "last_price","price_mean_all","price_median_all","price_std_all",
    "price_min_all","price_max_all","price_cv_all",
    "p_last5_mean","p_last5_median","p_last5_std","p_last5_min","p_last5_max",
    "p_last20_mean","p_last20_median","p_last20_std",
    "last_vs_mean","last_vs_min","momentum_short","momentum_long",
    "days_since_last","n_obs_all",
    "disc_freq","disc_mean","disc_max","last_discount",
    "shop_n_listings","shop_price_median","cat_median","cat_mean",
    "dow","dom","month","is_weekend","is_double_date",
    "day_avg_discount","day_promo_rate","day_free_ship","cat_disc_shrunk",
]
A3_CAT_FEATS = ["shopId","itemId","modelId","cat_id_hist","brand_hist"]
A3_ALL_FEATS = A3_NUM_FEATS + A3_CAT_FEATS

t0 = time.time()
model_a3 = train_approach3(train)
eval_a3  = evaluate_approach3(model_a3, train, validation)
for col, label in [("pred_raw","raw"),("pred_l1","+L1"),("pred_l2","+L2"),("pred_l3","+L3")]:
    print(f"A3 {label:5s}: {metrics(eval_a3[col], eval_a3['true_price'])}")
print(f"Runtime: {time.time()-t0:.1f}s")

def row(label, m):
    return {"approach": label, **m}

summary = pd.DataFrame([
    row("A1 raw",                      metrics(eval_a1["pred_raw"], eval_a1["true_price"])),
    row("A1 + calib",                  metrics(eval_a1["pred_cal"], eval_a1["true_price"])),
    row("A2 per-shop raw",             metrics(results_a2["Per-shop"]["pred_raw"],   results_a2["Per-shop"]["true_price"])),
    row("A2 per-shop + calib",         metrics(results_a2["Per-shop"]["pred_cal"],   results_a2["Per-shop"]["true_price"])),
    row("A2 per-product raw",          metrics(results_a2["Per-product"]["pred_raw"], results_a2["Per-product"]["true_price"])),
    row("A2 per-product + calib",      metrics(results_a2["Per-product"]["pred_cal"], results_a2["Per-product"]["true_price"])),
    row("A3 raw",                      metrics(eval_a3["pred_raw"], eval_a3["true_price"])),
    row("A3 + L1",                     metrics(eval_a3["pred_l1"], eval_a3["true_price"])),
    row("A3 + L2",                     metrics(eval_a3["pred_l2"], eval_a3["true_price"])),
    row("A3 + L3 (final)",             metrics(eval_a3["pred_l3"], eval_a3["true_price"])),
])
print("\n" + "=" * 60)
print("FINAL COMPARISON (validation days)")
print("=" * 60)
print(summary.to_string(index=False))

In [ ]:
model_path = Path("D:\Daftar Kerja\MrScrapper - AI Engineer\price_prediction\models")
path = model_path / "a3_global.lgb"
model_a3.save_model(str(path), num_iteration=model_a3.best_iteration)

In [ ]:
def predict_test_a3(test_df, train_df, model):
    test_df = test_df.copy()
    test_df["day"] = test_df["capturedAt"].dt.floor("D")
    out = []
    for day, day_rows in test_df.groupby("day"):
        anchors = day_rows[day_rows["price"].notna()].copy()
        targets = day_rows[day_rows["price"].isna()].copy()
        if len(anchors) == 0 or len(targets) == 0:
            continue
        print(f"  [{day.date()}] anchors={len(anchors)}, targets={len(targets):,}")

        tf = build_history_features(train_df, targets)
        af = build_history_features(train_df, anchors)
        tf = inject_anchor_state(tf, anchors, train_df)
        af = inject_anchor_state(af, anchors, train_df)
        tf = add_temporal_features(tf); af = add_temporal_features(af)

        # Cold-start fallback for last_price
        for f in (tf, af):
            mask = f["last_price"].isna() | (f["last_price"] <= 0)
            f.loc[mask, "last_price"] = (
                f.loc[mask, "shop_price_median"]
                 .fillna(f.loc[mask, "cat_median"])
                 .fillna(train_df["price"].median())
            )

        for c in A3_CAT_FEATS:
            tf[c] = tf[c].astype("category"); af[c] = af[c].astype("category")

        raw_t = tf["last_price"].values * np.exp(
            model.predict(tf[A3_ALL_FEATS], num_iteration=model.best_iteration))
        raw_a = af["last_price"].values * np.exp(
            model.predict(af[A3_ALL_FEATS], num_iteration=model.best_iteration))

        cal_fn, diag = fit_calibrator(raw_a, anchors["price"].values,
                                      cat_ids=af["cat_id_hist"].values)
        targets_out = targets.copy()
        targets_out["price"] = cal_fn(raw_t, cat_id_array=tf["cat_id_hist"].values)
        out.append(targets_out)
        print(f"     calibration global_bias={diag['global_bias']:+.3f}")

    anchors_all = test_df[test_df["price"].notna()].copy()
    sub = pd.concat([anchors_all] + out, ignore_index=True)
    return sub.drop(columns=["day"]).sort_values(["capturedAt"] + KEY)
